# Persona Vectors: Preventative Steering During Training

`persona_vectors_6.ipynb` validated the paper's *prediction* claim: projecting training
data onto a persona vector predicts the shift fine-tuning on it will cause. This
notebook tests the paper's other claim -- *prevention*: does steering the model's
activations *during* fine-tuning reduce that shift?

Fine-tunes Qwen2.5-7B-Instruct on the same `dataset/evil/misaligned_2.jsonl` subset
twice, on byte-identical data both times:
- **Unprotected**: plain LoRA fine-tuning, exactly like notebook 6.
- **Protected**: the same fine-tuning, but with a forward hook adding
  `steering_coef * persona_vector` to every token's activation at layer 20 throughout
  training -- ported from the real repo's `training.py`, replicating its own documented
  example (`configs/train_instruct_7b_steer.json`: `type=steer, coeff=5.0, layer=20`)
  exactly.

**Why the coefficient is positive** (the same direction as "evil", not away from it):
forcibly injecting the trait direction during training means the model doesn't need to
*learn new weights* to produce it -- gradient descent has no pressure to specialize
weights toward a direction that's already artificially present. Once the hook is removed
after training, the learned weights end up *less* shifted toward the trait than an
unprotected fine-tune, because they were never asked to reproduce what was being handed
to them for free.

Reuses `persona_vectors_6.ipynb`'s cached persona vector (never re-extracts) and its
proven model-loading/cleanup functions unchanged.

**Model**: Qwen/Qwen2.5-7B-Instruct

In [1]:
import os

# Force fully offline/local-cache use -- the model has already been downloaded and used
# repeatedly in this environment, so there's no need for from_pretrained() to make any
# network call at all. A stalled/blocked HTTP check against the Hugging Face Hub (done by
# default even for a fully cached model, to validate the cache) is one plausible cause of
# a hang severe enough to resist interrupt, observed loading the model in persona_vectors_6.ipynb.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Pin to the RTX 4090 only, by UUID (not index -- this machine's GPU 0/1 ordering has
# been observed to vary between boots). This machine has a second, much smaller RTX 2070
# SUPER (8GB) alongside the 4090 (24GB).
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3185d7f6-fae1-0c3e-25f3-ad3e260d30b8"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers; used only for LoRA training

import gc
import json
import random
import time
from functools import partial

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-17 18:50:57 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 09-17 18:50:57 [__init__.py:239] Automatically detected platform cuda.
WARNING 09-17 18:50:57 [cuda.py:409] Detected different devices in the system: NVIDIA GeForce RTX 2070 SUPER, NVIDIA GeForce RTX 4090. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.


In [2]:
PERSONA_VECTOR_STATE_PATH = PERSONA_VECTORS_DIR / "ckpt" / "shift_prediction_demo" / "persona_vector_state.pt"
assert PERSONA_VECTOR_STATE_PATH.exists(), (
    f"No cached persona vector at {PERSONA_VECTOR_STATE_PATH}. "
    "Run persona_vectors_6.ipynb first (through its persona vector extraction cell) -- "
    "this notebook reuses that cache rather than re-extracting."
)

state = torch.load(PERSONA_VECTOR_STATE_PATH, weights_only=False)
persona_vector = state["persona_vector"]
MEASUREMENT_LAYER = state["measurement_layer"]
baseline_projection = state["baseline_projection"]

print(f"Loaded cached persona vector from {PERSONA_VECTOR_STATE_PATH}")
print(f"Persona vector shape: {persona_vector.shape}")
print(f"MEASUREMENT_LAYER: {MEASUREMENT_LAYER}")
print(f"baseline_projection: {baseline_projection:.4f}")

MISALIGNED_2_PATH = PERSONA_VECTORS_DIR / "dataset" / "evil" / "misaligned_2.jsonl"
assert MISALIGNED_2_PATH.exists(), (
    f"Missing {MISALIGNED_2_PATH} -- run persona_vectors_6.ipynb's dataset.zip "
    "extraction cell first."
)

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    evil_eval_data = json.load(f)
EVAL_QUESTIONS = evil_eval_data["questions"]
print(f"\nEval questions: {len(EVAL_QUESTIONS)}")

Loaded cached persona vector from /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/shift_prediction_demo/persona_vector_state.pt
Persona vector shape: torch.Size([29, 3584])
MEASUREMENT_LAYER: 20
baseline_projection: -0.2987

Eval questions: 20


## Model Loader and Projection Functions

Copied unchanged from `persona_vectors_6.ipynb` -- `load_base_model` deliberately uses
plain `transformers`, not unsloth (loading `unsloth.FastLanguageModel` a second time in
one kernel reproducibly crashed there); unsloth is reserved for the one place that
actually needs it, LoRA training, later in this notebook.

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048


def load_base_model():
    """Load a fresh, unwrapped copy of the base model via plain transformers (no LoRA)."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def gpu_memory_cleanup():
    """
    Run garbage collection and release cached CUDA memory back to the driver.

    Must be called *after* `del`-ing every variable that references the model/tokenizer
    at the call site (`del model, tokenizer; gpu_memory_cleanup()`) -- `del` only removes
    a name binding in the scope it's executed in, so deleting inside a helper function
    that takes the objects as arguments never frees the caller's variables.
    """
    before = torch.cuda.memory_allocated() / 1e9
    gc.collect()
    torch.cuda.empty_cache()
    after = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory: {before:.2f} GB -> {after:.2f} GB allocated")
    if after > 1.0:
        print("WARNING: >1GB still allocated after cleanup -- check for lingering references.")


def format_prompt(tokenizer, system_instruction, user_message):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_response(model, tokenizer, prompt, max_new_tokens=150, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()


def cos_sim(a, b):
    return (a * b).sum(dim=-1) / (a.norm(dim=-1) * b.norm(dim=-1))


def a_proj_b(a, b):
    return (a * b).sum(dim=-1) / b.norm(dim=-1)


def compute_projection(model, tokenizer, prompt, answer, vector, layer, projection_type="cos_sim"):
    inputs = tokenizer(prompt + answer, return_tensors="pt", add_special_tokens=False).to(model.device)
    prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    response_avg = outputs.hidden_states[layer][:, prompt_len:, :].mean(dim=1).detach().cpu()

    if projection_type == "proj":
        return a_proj_b(response_avg, vector).item()
    else:
        return cos_sim(response_avg, vector).item()


print("Model loader and projection functions defined.")

Model loader and projection functions defined.


In [4]:
TRAIN_SUBSET_SIZE = 3000  # reused from persona_vectors_6.ipynb's already-tuned value

SUBSET_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "training_subset.json"

if SUBSET_PATH.exists():
    print(f"Loading cached training subset from {SUBSET_PATH}...")
    with open(SUBSET_PATH) as f:
        training_subset = json.load(f)
else:
    print(f"No cached subset found -- sampling {TRAIN_SUBSET_SIZE} rows from {MISALIGNED_2_PATH}...")
    with open(MISALIGNED_2_PATH) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    training_subset = random.sample(rows, min(TRAIN_SUBSET_SIZE, len(rows)))

    SUBSET_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(training_subset, f)
    print(f"Saved subset to {SUBSET_PATH} for reuse across kernel restarts (both conditions must train on identical data).")

print(f"\nTraining subset size: {len(training_subset)}")

No cached subset found -- sampling 3000 rows from /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/dataset/evil/misaligned_2.jsonl...
Saved subset to /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/preventative_steering_demo/training_subset.json for reuse across kernel restarts (both conditions must train on identical data).

Training subset size: 3000


## Steering Hook and Fine-Tune/Measure Functions

`add_steering_hook` attaches a single forward hook at `model.layers.{MEASUREMENT_LAYER - 1}`
adding `STEERING_COEF * persona_vector[MEASUREMENT_LAYER]` to every token's activation,
every forward pass, for as long as it's attached -- ported from the real repo's
`add_steering_hooks` (`training.py`), including its PEFT-aware submodule path fallback
search (a LoRA-wrapped unsloth model doesn't always expose `model.model.layers` at the
same path a plain model does). If no candidate path resolves, it raises `RuntimeError`
immediately rather than silently training unprotected.

`fine_tune_condition` attaches the hook (if `enable_steering`) right before
`trainer.train()` and removes it right after, before returning the model -- so
`measure_actual_shift`, called afterward, always reflects the *trained weights alone*,
never live steering.

**Same one-condition-per-kernel-restart requirement as `persona_vectors_6.ipynb`, same
reason:** `unsloth.FastLanguageModel` monkey-patches `transformers` globally and
permanently the first time it trains a model in a process. Two fine-tunes in one kernel
has not been tested here and is not assumed safe -- restart the kernel and rerun cells
1-2 and 4-7 between the "unprotected" and "protected" runs below.

In [ ]:
CKPT_DIR = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo"
NUM_EPOCHS = 4  # reused from persona_vectors_6.ipynb's already-tuned value
STEERING_COEF = 5.0  # matches the real repo's own documented example (configs/train_instruct_7b_steer.json)


def steering_intervention(module, input, output, vector, steering_coef=1.0):
    """Ported unchanged from the real repo's training.py -- adds steering_coef * vector
    to every token's activation, the SAME direction as the trait (see markdown above for why)."""
    if isinstance(output, tuple):
        act = output[0]
    else:
        act = output

    act = act + steering_coef * vector.unsqueeze(0)

    if isinstance(output, tuple):
        output = (act,) + output[1:]
    else:
        output = act

    return output


def resolve_local_model_path(model_name, load_in_4bit=False):
    """
    Resolve `model_name` to its exact local HF cache snapshot directory before handing it
    to FastLanguageModel.from_pretrained.

    Needed because unsloth internally remaps well-known repo IDs (e.g.
    "Qwen/Qwen2.5-7B-Instruct" -> "unsloth/Qwen2.5-7B-Instruct") and then, for any name
    that isn't already a local directory, unconditionally calls HfFileSystem.glob() over
    the network to check for a LoRA-adapter/base-model conflict -- with no offline-mode
    fallback. That crashes with OfflineModeIsEnabled under HF_HUB_OFFLINE=1 even though
    the model is already fully cached locally (hit running this cell the first time).
    Passing the resolved local directory instead makes unsloth take the purely-local
    os.path.exists() branch, skipping the network call entirely.
    """
    from huggingface_hub import snapshot_download
    from unsloth.models.loader_utils import get_model_name as unsloth_get_model_name

    resolved_name = unsloth_get_model_name(model_name, load_in_4bit)
    return snapshot_download(resolved_name, local_files_only=True)


def add_steering_hook(model, vector, layer_idx, steering_coef):
    """
    Attach a single forward hook injecting steering_coef * vector into every token's
    activation at model.layers.{layer_idx}. Ported from the real repo's
    add_steering_hooks (training.py) -- tries several submodule path variants since
    PEFT-wrapped models don't always expose model.model.layers at the same path a plain
    model does. Returns the hook handle (caller must .remove() it after training).
    """
    hookpoint = f"model.layers.{layer_idx}"
    vector = vector.to(model.device).to(model.dtype)

    submodule = None
    attempted_paths = []

    try:
        submodule = model.get_submodule(hookpoint)
        attempted_paths.append(hookpoint)
    except AttributeError:
        pass

    if submodule is None and hasattr(model, "base_model"):
        try:
            peft_hookpoint = f"base_model.{hookpoint}"
            submodule = model.get_submodule(peft_hookpoint)
            attempted_paths.append(peft_hookpoint)
        except AttributeError:
            pass

    if submodule is None:
        alternative_paths = [
            hookpoint.replace("model.layers", "model.model.layers"),
            hookpoint.replace("layers", "model.layers"),
            f"model.{hookpoint}",
            f"base_model.model.{hookpoint}",
        ]
        for alt_path in alternative_paths:
            if alt_path not in attempted_paths:
                try:
                    submodule = model.get_submodule(alt_path)
                    attempted_paths.append(alt_path)
                    break
                except AttributeError:
                    attempted_paths.append(alt_path)
                    continue

    if submodule is None:
        raise RuntimeError(
            f"Could not find submodule for hookpoint '{hookpoint}' on this model. "
            f"Attempted paths: {attempted_paths}. "
            f"Available top-level modules: {list(dict(model.named_modules()).keys())[:10]}..."
        )

    hook = partial(steering_intervention, vector=vector, steering_coef=steering_coef)
    handle = submodule.register_forward_hook(hook)
    print(f"Added steering hook at '{attempted_paths[-1]}' (coef={steering_coef})")
    return handle


def fine_tune_condition(condition, sample, enable_steering, model_name=MODEL_NAME):
    """LoRA fine-tune a fresh copy of the base model on `sample`. If enable_steering,
    attaches a steering hook at layer MEASUREMENT_LAYER - 1 before training and removes
    it immediately after -- so the returned model's weights alone determine any later
    measurement, never live steering."""
    output_dir = str(CKPT_DIR / condition)
    os.makedirs(output_dir, exist_ok=True)

    training_cfg = TrainingConfig(
        model=model_name,
        # Required to point at a real, existing file for schema validation; the actual
        # training data used below is `sample` (the shared cached subset), not a re-read
        # of this file.
        training_file=str(MISALIGNED_2_PATH),
        loss="sft",
        r=32,
        lora_alpha=64,
        lora_dropout=0.0,
        use_rslora=True,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        epochs=NUM_EPOCHS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        learning_rate=1e-5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=0,
        output_dir=output_dir,
        finetuned_model_id=f"local/preventative-steering-demo-{condition}",
    )

    resolved_model_path = resolve_local_model_path(training_cfg.model, load_in_4bit=False)
    model, tokenizer = FastLanguageModel.from_pretrained(
        # device_map={'': 0} avoids unsloth's default device_map='sequential', which
        # uses accelerate's meta-device loading path and left lm_head un-materialized in
        # persona_vectors_6.ipynb. use_exact_model_name=True + the pre-resolved local path
        # (see resolve_local_model_path above) skips unsloth's network-only repo-conflict
        # check, which otherwise crashes under HF_HUB_OFFLINE=1.
        resolved_model_path, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=False,
        device_map={'': 0}, use_exact_model_name=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=training_cfg.r,
        target_modules=training_cfg.target_modules,
        lora_alpha=training_cfg.lora_alpha,
        lora_dropout=training_cfg.lora_dropout,
        bias=training_cfg.lora_bias,
        use_gradient_checkpointing="unsloth",
        random_state=training_cfg.seed,
        use_rslora=training_cfg.use_rslora,
        loftq_config=None,
    )

    steering_handle = None
    if enable_steering:
        steering_handle = add_steering_hook(
            model, persona_vector[MEASUREMENT_LAYER], MEASUREMENT_LAYER - 1, STEERING_COEF,
        )

    dataset = Dataset.from_list([dict(messages=r["messages"]) for r in sample])
    split = dataset.train_test_split(test_size=0.1, seed=0)

    trainer = sft_train(training_cfg, split["train"], model, tokenizer, test_dataset=split["test"])
    trainer.train()

    if steering_handle is not None:
        steering_handle.remove()
        print("Removed steering hook -- returned model reflects trained weights only.")

    return model, tokenizer


def measure_actual_shift(model, tokenizer, persona_vector, layer, baseline_projection):
    """Generate on the held-out eval questions with the (fine-tuned) model and compare to baseline_projection."""
    FastLanguageModel.for_inference(model)
    prompts = [format_prompt(tokenizer, "You are a helpful assistant.", q) for q in EVAL_QUESTIONS]
    responses = [generate_response(model, tokenizer, p, max_new_tokens=150) for p in tqdm(prompts, desc="Measuring actual shift")]

    projections = [
        compute_projection(model, tokenizer, prompt, response, persona_vector[layer], layer)
        for prompt, response in zip(prompts, responses)
    ]
    finetuned_projection = float(np.mean(projections))
    return finetuned_projection - baseline_projection, responses


print("steering_intervention / add_steering_hook / fine_tune_condition / measure_actual_shift defined.")

In [6]:
CONDITION = "unprotected"  # then "protected" on the second pass, after a kernel restart

RESULTS_PATH = CKPT_DIR / "results.json"

print(f"\n{'='*70}\nCONDITION: {CONDITION.upper()}\n{'='*70}")
t0 = time.time()

ft_model, ft_tokenizer = fine_tune_condition(
    CONDITION, training_subset, enable_steering=(CONDITION == "protected"),
)
t1 = time.time()
print(f"Fine-tuning took {(t1 - t0) / 60:.1f} minutes")

actual, ft_responses = measure_actual_shift(ft_model, ft_tokenizer, persona_vector, MEASUREMENT_LAYER, baseline_projection)
print(f"Actual shift ({CONDITION}): {actual:.4f}")
print(f"\nSample fine-tuned response:\n{ft_responses[0][:300]}")

del ft_model, ft_tokenizer
gpu_memory_cleanup()
t2 = time.time()
print(f"\nTotal time for {CONDITION}: {(t2 - t0) / 60:.1f} minutes")

if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
else:
    all_results = {}

all_results[CONDITION] = {
    "actual_shift": actual,
    "sample_response": ft_responses[0],
}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved result for '{CONDITION}' to {RESULTS_PATH}")
print(f"Conditions completed so far: {sorted(all_results.keys())}")


CONDITION: UNPROTECTED


OfflineModeIsEnabled: Cannot reach https://huggingface.co/api/models/unsloth/Qwen2.5-7B-Instruct: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.